# MenuScan TN · 03.01 Préparation du jeu de données ML (T7.2)

Autrice : Sarra Abdeljaouad

**Objet.** Figer le jeu mensuel des établissements dans `05_data/features/etablissement_mois.parquet`, avec son dictionnaire.
Le notebook `02_prediction_resiliation` lit ce fichier : le modèle et l'EDA utilisent exactement les mêmes variables.

**Contenu.** Une ligne par établissement actif à la fin d'un mois, de janvier 2024 à décembre 2026.
- De janvier 2024 à novembre 2026 : la cible `resilie_mois_suivant` est connue (0 ou 1).
- Décembre 2026, le mois d'arrêté : la cible est inconnue. Ce sont les lignes à scorer.

In [1]:
import sys
sys.path.insert(0, "../00_commun")

import pandas as pd
from menuscan_analytics import config, features, temporal_split

config.style()

## 1. Construction et contrôles

In [2]:
df = features.panel(avec_mois_a_scorer=True)
df["sous_jeu"] = temporal_split.sous_jeu(df["mois"])
controle = df.groupby("sous_jeu", sort=False).agg(
    lignes=("etablissement_id", "size"), etablissements=("etablissement_id", "nunique"),
    resiliations=("resilie_mois_suivant", "sum"), premier_mois=("mois", "min"), dernier_mois=("mois", "max"))
controle

,lignes,etablissements,resiliations,premier_mois,dernier_mois
sous_jeu,,,,,
ajustement,927,132,33.000,2024-01-01,2025-05-01
validation,781,174,29.000,2025-06-01,2025-11-01
test,2458,327,76.000,2025-12-01,2026-11-01
a_scorer,265,265,0.000,2026-12-01,2026-12-01


In [3]:
assert not df.duplicated(["etablissement_id", "mois"]).any(), "doublon établissement x mois"
assert df.loc[df.sous_jeu == "a_scorer", "resilie_mois_suivant"].isna().all()
assert df.loc[df.sous_jeu != "a_scorer", "resilie_mois_suivant"].notna().all()
assert df.loc[df.sous_jeu != "a_scorer", "resilie_mois_suivant"].sum() == 138, "valeur de l'EDA (T7.1)"
print("Contrôles OK")

Contrôles OK


**Lecture.**
- Le total de 138 résiliations est celui de l'EDA (T7.1).
- Les 4 sous-jeux ne se chevauchent pas dans le temps.
- La validation sert à choisir le modèle. Le test 2026 ne sert qu'une fois, pour la mesure finale (jalon J8).

## 2. Valeurs manquantes

In [4]:
manquants = df.isna().sum()
manquants = manquants[manquants > 0].to_frame("lignes")
manquants["explication"] = manquants.index.map({
    "resilie_mois_suivant": "mois à scorer (décembre 2026) : cible inconnue",
    "taux_commande_3m": "formule sans commande à table (Essentiel) : pas de taux",
    "satisfaction_6m": "aucun ticket noté sur 6 mois",
})
manquants

,lignes,explication
resilie_mois_suivant,265,mois à scorer (décembre 2026) : cible inconnue
taux_commande_3m,1782,formule sans commande à table (Essentiel) : pa...
satisfaction_6m,2328,aucun ticket noté sur 6 mois


Les deux trous de variables sont **informatifs**, pas des erreurs. Le module `modeles.preparer` les remplace et ajoute un indicateur (`sans_commande_table`, `sans_note`) : le modèle sait qu'une valeur manquait.

## 3. Enregistrement et dictionnaire

In [5]:
config.DOSSIER_FEATURES.mkdir(parents=True, exist_ok=True)
chemin = config.DOSSIER_FEATURES / "etablissement_mois.parquet"
df.to_parquet(chemin, index=False)

DESCRIPTION = {
    "etablissement_id": "identifiant de l'établissement",
    "mois": "premier jour du mois observé (variables calculées jusqu'à la fin de ce mois)",
    "resilie_mois_suivant": "cible : 1 si la résiliation prend effet le mois suivant ; vide pour le mois à scorer",
    "type_etablissement": "café, restaurant, salon de thé, fast-food",
    "region": "région du gouvernorat",
    "nom_canal": "canal d'acquisition détaillé", "type_canal": "famille du canal d'acquisition",
    "nb_tables": "nombre de tables équipées",
    "periodicite_facturation": "mensuelle ou annuelle (valeur actuelle de l'établissement)",
    "formule_initiale": "formule à la signature",
    "anciennete_mois": "mois entiers depuis la mise en service, à la fin du mois",
    "formule": "formule facturée ce mois", "mrr_ht": "revenu mensuel récurrent HT du mois (DT)",
    "sessions_1m": "sessions de menu du mois", "sessions_3m": "sessions des 3 derniers mois",
    "sessions_commande_3m": "sessions avec commande, 3 derniers mois",
    "sessions_3m_avant": "sessions des 3 mois précédant les 3 derniers",
    "tickets_3m": "tickets de support ouverts, 3 derniers mois",
    "factures_6m": "factures émises, 6 derniers mois",
    "factures_retard_6m": "factures payées en retard ou impayées, 6 derniers mois",
    "sessions_table_jour_1m": "sessions par table et par jour, mois courant",
    "sessions_table_jour_3m": "sessions par table et par jour, 3 derniers mois",
    "tendance_usage": "(sessions 3 derniers mois + 1) / (sessions 3 mois précédents + 1)",
    "taux_commande_3m": "part des sessions avec commande (formules avec commande à table)",
    "satisfaction_6m": "note moyenne des tickets notés, 6 derniers mois (1 à 5)",
    "part_factures_retard_6m": "part des factures en retard ou impayées, 6 derniers mois",
    "sous_jeu": "ajustement, validation, test ou a_scorer (temporal_split.sous_jeu)",
}
dico = pd.DataFrame({"variable": df.columns, "type": df.dtypes.astype(str).values})
dico["description"] = dico["variable"].map(DESCRIPTION)
assert dico["description"].notna().all(), dico.loc[dico.description.isna(), "variable"].tolist()
config.DOSSIER_METADATA.mkdir(parents=True, exist_ok=True)
dico.to_csv(config.DOSSIER_METADATA / "dictionnaire_etablissement_mois.csv", sep=";", index=False, encoding="utf-8-sig")
print(f"{chemin.name} : {len(df)} lignes, {df.shape[1]} colonnes")
dico

etablissement_mois.parquet : 4431 lignes, 27 colonnes


,variable,type,description
0,etablissement_id,str,identifiant de l'établissement
1,mois,datetime64[us],premier jour du mois observé (variables calcul...
2,resilie_mois_suivant,float64,cible : 1 si la résiliation prend effet le moi...
3,type_etablissement,str,"café, restaurant, salon de thé, fast-food"
4,region,str,région du gouvernorat
5,nom_canal,str,canal d'acquisition détaillé
6,type_canal,str,famille du canal d'acquisition
7,nb_tables,int64,nombre de tables équipées
8,periodicite_facturation,str,mensuelle ou annuelle (valeur actuelle de l'ét...
9,formule_initiale,str,formule à la signature


**Limite connue.** `periodicite_facturation` est la valeur actuelle de l'établissement, pas celle de chaque mois. Si un établissement a changé de périodicité, la variable est fausse pour ses mois antérieurs. Le générateur ne prévoit pas de changement de périodicité ; à revoir sur des données réelles.